### **Libraries**

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
import time
import databento as db
import pandas as pd
import download_functions       # .py file with the download functions
import warnings
from databento.common.error import BentoWarning

# Silence weekend / degraded-data warnings
warnings.filterwarnings("ignore", category=BentoWarning)

### **Run code**

In [2]:
df = download_functions.pull_all_settlement_data()
df.head()

[fx] 6A: checkpoint found, loading from disk.
[fx] 6C: checkpoint found, loading from disk.
[fx] 6B: checkpoint found, loading from disk.
[fx] 6E: checkpoint found, loading from disk.
[fx] 6J: checkpoint found, loading from disk.
[fx] 6S: checkpoint found, loading from disk.
[energy] CL: checkpoint found, loading from disk.
[energy] HO: checkpoint found, loading from disk.
[energy] NG: checkpoint found, loading from disk.
[energy] RB: checkpoint found, loading from disk.
[metals] GC: checkpoint found, loading from disk.
[metals] SI: checkpoint found, loading from disk.
[metals] HG: checkpoint found, loading from disk.
[metals] PL: checkpoint found, loading from disk.
[metals] PA: checkpoint found, loading from disk.
[rates] ZT: checkpoint found, loading from disk.
[rates] ZF: checkpoint found, loading from disk.
[rates] ZN: checkpoint found, loading from disk.
[rates] ZB: checkpoint found, loading from disk.
[equity_index] ES: checkpoint found, loading from disk.
[equity_index] NQ: che

,trade_date,root,category,raw_symbol,instrument_id,expiration,settlement_price,open_interest
17424,2019-08-19,6A,fx,6AF0,216474,2020-01-13 15:16:00+00:00,0.6792,NaN
17425,2019-08-20,6A,fx,6AF0,216474,2020-01-13 15:16:00+00:00,0.6801,NaN
17426,2019-08-21,6A,fx,6AF0,216474,2020-01-13 15:16:00+00:00,0.6806,NaN
17427,2019-08-22,6A,fx,6AF0,216474,2020-01-13 15:16:00+00:00,0.6784,NaN
17428,2019-08-23,6A,fx,6AF0,216474,2020-01-13 15:16:00+00:00,0.6778,NaN


### **Data description**

In [3]:
contracts_per_root = df.groupby("root").agg(
    category=("category", "first"),
    n_contracts=("instrument_id", "nunique"),
    n_rows=("trade_date", "size"),
    first_obs=("trade_date", "min"),
    last_obs=("trade_date", "max"),
).sort_values("category")

contracts_per_root

,category,n_contracts,n_rows,first_obs,last_obs
root,,,,,
GF,ags_livestock,132,33027,2010-06-04,2026-06-30
HE,ags_livestock,132,46781,2010-06-04,2026-06-30
LE,ags_livestock,99,35425,2010-06-04,2026-06-30
ZC,ags_livestock,84,57011,2010-06-04,2026-06-30
ZL,ags_livestock,133,90256,2010-06-04,2026-06-30
ZS,ags_livestock,117,77405,2010-06-04,2026-06-30
ZW,ags_livestock,85,50742,2010-06-04,2026-06-30
CL,energy,198,261008,2010-06-04,2026-06-30
HO,energy,198,150728,2010-06-04,2026-06-30


### **Check for missing data**

In [4]:
missing_settlement = df.groupby("root").agg(
    n_rows=("settlement_price", "size"),
    n_missing_settlement=("settlement_price", lambda x: x.isna().sum()),
    n_missing_oi=("open_interest", lambda x: x.isna().sum()),
)
missing_settlement["pct_missing_settlement"] = (
    missing_settlement["n_missing_settlement"] / missing_settlement["n_rows"] * 100
).round(2)
missing_settlement["pct_missing_oi"] = (
    missing_settlement["n_missing_oi"] / missing_settlement["n_rows"] * 100
).round(2)

missing_settlement.sort_values("pct_missing_settlement", ascending=False)

,n_rows,n_missing_settlement,n_missing_oi,pct_missing_settlement,pct_missing_oi
root,,,,,
LE,35425,141,12653,0.40,35.72
GF,33027,127,11870,0.38,35.94
RTY,11300,43,5443,0.38,48.17
HE,46781,179,17858,0.38,38.17
PL,30935,116,14340,0.37,46.36
GC,79898,293,34158,0.37,42.75
ZC,57011,203,22115,0.36,38.79
SI,75028,260,35628,0.35,47.49
NQ,22606,79,10507,0.35,46.48


**Comments**

- The percentage of missing settlement prices is considerably low. Most of these missing prices are in contracts with long expiries, so these missing prices will not affect the front month contract roll. THud, we suggest to forward fill settlement prices if needed.

- The percentage of missing Open Interest is quite large. However, most of these missing stats are in contracts that just started trading and expiration is far away. In other words, even though the contract was availabe for trading, there were no trades or open interest at the moment, reason why it was not reported.


### **Save data**

In [5]:
df.to_parquet(download_functions.OUTPUT_PATH, index=False)